# 021 — D anchors with constrained boundary expansion + 020 H/A

Start with automatic high-precision broadcast D cores from 017: bridge gaps <=10 frames, keep runs >=100 frames. Each core defines a candidate stoppage. A contiguous interval around it is selected by maximizing accumulated D-vs-LIVE evidence from frozen 016 probabilities and tactical ball/player movement. No new D island can appear without a core from one of the two automatic sources.

Для каждого ядра выбираются левая и правая границы по максимальной накопленной сумме свидетельств. Это оптимизация одного непрерывного D-интервала с фиксированной серединой. Расширение ограничено радиусом 2/5/10/15 с, соседними ядрами и устойчивым барьером LIVE (P(D)<.15, мяч >=3 м/с, игроки >=1 м/с в течение .6 с). Пропуски движения нейтральны. Модельная оценка logit(P(D)) относительно порога дополняется штрафом за движение и поддержкой неподвижности, сглаживается на 1 с внутри рассматриваемого окна. Это эвристические свидетельства, не новая калиброванная вероятность.

122 candidate configurations: two seed sources × (unexpanded cores plus 4 radii × 5 evidence thresholds × 3 motion weights). Second source adds frozen 016 P(D)>=.9 runs lasting at least 1 second with the same 10-frame gap closure. Choose maximum D recall subject to precision >=.99 (main) and >=.95 (diagnostic variant), at least 100 predicted D frames, on first 4 public minutes ONLY. Both the weak cores and evidence are recomputed inside that isolated prefix; no future validation core enters parameter selection. Freeze choices before evaluating minute 5. 016 itself was trained on the public prefix, which has also been repeatedly examined: this is in-sample tuning and repeated diagnostics, not an independent test.

Frozen H/A and cleanup from 020, including the penalty-area gate and no-X fallback. D never crosses half boundaries. No neural training, manual annotation, private truth, or hard-coded match-specific repairs. Offline full-match inference uses future input observations, not future labels. Weak cores can be wrong; all their D frames are retained, so false anchors cannot be corrected by this experiment.

Run All after preparing 020/016 features and broadcast_alignment (clean preparation notebook in sol/010_dead_logistic). Only solution code/configuration is published. Predicted CSVs, traces, metrics and executed notebook stay local_review/experiments/021_dead_anchor_expansion. Required libraries: NumPy, pandas, scikit-learn, PyTorch, tqdm, Jupyter, matplotlib, standard library. Code reuses parent notebook preparation definitions without fitting or parameter selection. The published selected_config.json records reproducible main and diagnostic choices; Run All recomputes them deterministically on permitted public labels.

Исходный вариант только с broadcast сохранён локально как broadcast_only_trial: обнаружено ограничение покрытия двумя из четырёх остановок, после чего добавлен второй источник. Поэтому расширенное исследование явно является адаптивной диагностикой на уже просмотренном публичном окне, а не новым независимым тестом.

Изоляция префикса относится к построению ядер и выбору границ в этом эксперименте. Кэшированные вероятности 016 и broadcast-сигналы сохраняют исходный офлайн-контекст входных данных; это не строго причинный тест. Эталонные метки пятой минуты в подбор не входят.


In [ ]:
from pathlib import Path
import json,sys,hashlib,itertools,time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists());sys.path.insert(0,str(ROOT))
from games import GAME,half_video_blocks
from evaluation import possession_metrics as pm
EXPERIMENT='021_dead_anchor_expansion';OUT=ROOT/'local_review/experiments'/EXPERIMENT;OUT.mkdir(parents=True,exist_ok=True)
N=199319;FPS=25;START=11185;SPLIT=17185;END=18685;BLOCKS=half_video_blocks(GAME)
def save_json(path,value):path.write_text(json.dumps(value,ensure_ascii=False,indent=2))
def progress(stage,**kw):save_json(OUT/'progress.json',dict(stage=stage,**kw))
# All probability/feature inputs are automatic; no reference labels are used here.
# Все входные вероятности и признаки автоматические; эталон здесь не используется.
probability=np.load(ROOT/'local_review/experiments/020_penalty_gated_corridor/p_D.npy')
assert probability.shape==(N,) and np.isfinite(probability).all()
assert np.allclose(probability,np.load(ROOT/'local_review/experiments/016_dead_broadcast_features/public_p_D.npy'),atol=1e-6)
features=np.load(ROOT/'local_review/experiments/008_weak_dead_boosting/features.npy',mmap_mode='r')
feature_names=json.loads((ROOT/'local_review/experiments/008_weak_dead_boosting/feature_names.json').read_text())
ball_speed=features[:,feature_names.index('ball_speed_25')]
player_speed=features[:,feature_names.index('median_player_speed')]
weak_broadcast=np.load(ROOT/'local_review/broadcast_alignment/weak_D_candidates.npy')
assert weak_broadcast.shape==(N,)
def spans(values):
 edges=np.r_[0,np.flatnonzero(values[1:]!=values[:-1])+1,len(values)]
 return list(zip(edges[:-1],edges[1:])) if len(values) else []
def duration_filter(raw,min_length=100):
 out=raw.copy()
 for a,b in spans(raw):
  if not raw[a] and a>0 and b<len(raw) and b-a<=10:out[a:b]=True
 for a,b in spans(out.copy()):
  if out[a] and b-a<min_length:out[a:b]=False
 return out

def moving_average(values,width=25):
 # Never borrow input frames beyond the current inference/selection interval.
 # Не используем кадры за границами текущего интервала подбора или инференса.
 indices=np.arange(len(values));lo=np.maximum(0,indices-width//2);hi=np.minimum(len(values),indices+width//2+1)
 sums=np.r_[0.,np.cumsum(values,dtype=np.float64)]
 return (sums[hi]-sums[lo])/(hi-lo)

def live_barrier(prob,ball,players):
 raw=(prob<.15)&(ball>=3)&(players>=1)
 result=np.zeros(len(prob),bool)
 for a,b in spans(raw):
  if raw[a] and b-a>=15:result[a:b]=True
 return result

def expand_cores(cores,evidence,barrier,radius):
 result=cores.copy();trace=[]
 anchors=[(a,b) for a,b in spans(cores) if cores[a]]
 for index,(a,b) in enumerate(anchors):
  left=max(0,a-radius,anchors[index-1][1] if index else 0)
  right=min(len(cores),b+radius,anchors[index+1][0] if index+1<len(anchors) else len(cores))
  blockers=np.flatnonzero(barrier[left:a])
  if len(blockers):left=left+int(blockers[-1])+1
  blockers=np.flatnonzero(barrier[b:right])
  if len(blockers):right=b+int(blockers[0])
  # Prefix gains give the exact best boundary for one contiguous interval containing the core.
  # Накопленные суммы дают точную лучшую границу непрерывного интервала, содержащего ядро.
  left_gain=np.r_[0.,np.cumsum(evidence[left:a][::-1])]
  right_gain=np.r_[0.,np.cumsum(evidence[b:right])]
  lo=a-int(np.argmax(left_gain));hi=b+int(np.argmax(right_gain))
  result[lo:hi]=True
  trace.append(dict(core_start=a,core_end=b,start=lo,end=hi))
 assert np.all(~cores|result)
 # Every predicted component must be anchored; no new unanchored false-alarm islands.
 # Каждый итоговый D-фрагмент обязан содержать ядро: новых островков без ядра нет.
 assert all(cores[a:b].any() for a,b in spans(result) if result[a])
 return result,trace

# Behavior checks are independent of this match and reference labels.
# Проверки логики независимы от матча и эталонных меток.
seed=np.zeros(40,bool);seed[15:20]=True
score=np.full(40,-2.);score[5:30]=1.
p,_=expand_cores(seed,score,np.zeros(40,bool),20)
assert np.array_equal(np.flatnonzero(p),np.arange(5,30))
bar=np.zeros(40,bool);bar[10:12]=True
p,_=expand_cores(seed,score,bar,20);assert np.array_equal(np.flatnonzero(p),np.arange(12,30))
p,_=expand_cores(seed,np.full(40,-1.),np.zeros(40,bool),20);assert np.array_equal(p,seed)
p,_=expand_cores(np.zeros(40,bool),score,np.zeros(40,bool),20);assert not p.any()
p,_=expand_cores(seed,score,np.zeros(40,bool),0);assert np.array_equal(p,seed)

def infer_window(config,first,stop):
 core=duration_filter(weak_broadcast[first:stop],100)
 if config['seed_source']=='broadcast_plus016':
  # A second source covers stoppages never shown as broadcast cutaways.
  # Второй источник покрывает остановки без подходящего плана трансляции.
  core |= duration_filter(probability[first:stop]>=.9,25)
 p=np.clip(probability[first:stop],.001,.999)
 ball=ball_speed[first:stop];players=player_speed[first:stop]
 moving=(ball>=3)&(players>=1)
 stationary=(ball<=.5)&(players<=.7)
 cutoff=config['evidence_threshold']
 evidence=np.log(p/(1-p))-np.log(cutoff/(1-cutoff))
 evidence+=config['motion_weight']*(.5*stationary.astype(float)-moving.astype(float))
 evidence=moving_average(evidence)
 mask,trace=expand_cores(core,evidence,live_barrier(p,ball,players),int(config['radius_seconds']*FPS))
 return mask,core,trace

def binary_scores(y,p):
 tp=int((y&p).sum());fp=int((~y&p).sum());fn=int((y&~p).sum());tn=int((~y&~p).sum())
 precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1);f1=2*tp/max(2*tp+fp+fn,1)
 return dict(precision=precision,recall=recall,F1=f1,predicted_frames=tp+fp,true_frames=tp+fn,
             true_positive=tp,false_positive=fp,false_negative=fn,false_D_seconds=fp/FPS)


In [ ]:
# Isolated prefix selection: future minute-five cores are not supplied to infer_window.
# Изолированный префикс: ядра пятой минуты не передаются в infer_window при подборе.
progress('boundary search')
gt=pm.read_label_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')
y_prefix=gt.set_index('frame').loc[np.arange(START,SPLIT),'label'].eq('D').to_numpy()
configs=[]
for source in ['broadcast','broadcast_plus016']:
 configs.append(dict(seed_source=source,radius_seconds=0,evidence_threshold=.5,motion_weight=0.))
 configs += [dict(seed_source=source,radius_seconds=r,evidence_threshold=t,motion_weight=w) for r,t,w in itertools.product([2,5,10,15],[.2,.35,.5,.65,.8],[0.,.75,1.5])]
search=[]
for index,config in enumerate(tqdm(configs,desc='D boundaries / Границы остановок')):
 prediction,cores,trace=infer_window(config,START,SPLIT)
 search.append(dict(candidate=index,**config,**binary_scores(y_prefix,prediction)))
selected_expansion={}
for target in [.99,.95]:
 feasible=[row for row in search if row['precision']>=target and row['predicted_frames']>=100]
 assert feasible, 'Original seed baseline must remain feasible on this development prefix'
 winner=max(feasible,key=lambda row:(row['recall'],row['precision'],-row['radius_seconds'],-row['motion_weight']))
 selected_expansion[str(target)]=configs[winner['candidate']]
# Save selected parameters before evaluating or printing any minute-five results.
# Сохраняем параметры ДО оценки или вывода результатов пятой минуты.
save_json(OUT/'selected_config.json',dict(primary_precision_target=.99,selection_range=[START,SPLIT],configs=selected_expansion,
 selection_context='isolated prefix; no future validation anchors; 016 trained on prefix; repeated diagnostic labels'))
pd.DataFrame(search).to_csv(OUT/'boundary_search_prefix.csv',index=False)
np.save(OUT/'p_D.npy',probability)
full_masks={};all_cores={};traces=[]
for target,config in selected_expansion.items():
 mask=np.zeros(N,bool);core_mask=np.zeros(N,bool)
 for half,first,last in BLOCKS:
  part,core,trace=infer_window(config,first,last+1);mask[first:last+1]=part;core_mask[first:last+1]=core
  for row in trace:traces.append(dict(target=target,half=half,**{key:int(value+first) for key,value in row.items()}))
 full_masks[target]=mask;all_cores[target]=core_mask
 np.save(OUT/f'dead_mask_p{int(float(target)*100)}.npy',mask)
old_seeds=ROOT/'local_review/experiments/017_dead_broadcast_agreement/cores_p99_dead_mask.npy'
broadcast_cores=np.zeros(N,bool)
for _,first,last in BLOCKS:broadcast_cores[first:last+1]=duration_filter(weak_broadcast[first:last+1],100)
if old_seeds.exists():assert np.array_equal(broadcast_cores,np.load(old_seeds))
full_cores=all_cores['0.99']
np.save(OUT/'core_mask.npy',full_cores);pd.DataFrame(traces).to_csv(OUT/'expansion_traces.csv',index=False)
filtered=full_masks['0.99'];np.save(OUT/'dead_mask.npy',filtered)
progress('H/A composition')


In [ ]:
# Run preparation and algorithm definitions only, keeping all outputs in 019.
# Запускаем только подготовку и определения 007; все выходы остаются в 019.
progress('007 preparation')
parent007=json.loads((ROOT/'sol/007_clean_anchors/experiment.ipynb').read_text())
for index in [1,3,4,6,8]:
 source=''.join(parent007['cells'][index]['source'])
 if index==1:
  source=source.replace('EXPERIMENT = "007_clean_anchors"','EXPERIMENT = "021_dead_anchor_expansion"')
  source=source.replace('ROOT / "sol" / EXPERIMENT / "grid.json"','ROOT / "sol/007_clean_anchors/grid.json"')
  source=source.replace('import matplotlib.pyplot as plt\n','')
 exec(compile(source,f'007 cell {index}','exec'))
# Gate the direct corridor assignment by the CHOSEN team's own defensive line.
# Прямое правило коридора разрешено только по линии ВЫБРАННОЙ команды.
# The line must be strictly beyond its own penalty-area depth, toward midfield.
# Линия должна быть строго дальше границы своей штрафной в сторону центра поля.
PENALTY_DEPTH_M=16.5
BOUNDARY_TOLERANCE_M=1e-9
def line_is_advanced(line_x,defends_left,length):
 depth=np.asarray(line_x)*length if defends_left else (1-np.asarray(line_x))*length
 return np.isfinite(depth)&(depth>PENALTY_DEPTH_M+BOUNDARY_TOLERANCE_M)
# Mirror-invariant checks: inside, exactly on boundary, beyond, and missing.
# Проверки симметрии сторон: внутри, на границе, снаружи и пропуск.
probe=np.array([10.,16.5,16.51,30.,np.nan])/LENGTH
for left in [True,False]:
 test=probe if left else 1-probe
 assert np.array_equal(line_is_advanced(test,left,LENGTH),[False,False,True,True,False])
original_outside_owner=outside_owner.copy()
line_eligible=np.zeros((N,2),bool)
for half,first,last in BLOCKS:
 for team in [0,1]:
  line_eligible[first:last+1,team]=line_is_advanced(last_outfield[first:last+1,team],sides[half][team]=='left',LENGTH)
had_direct=outside_owner!=3
rows=np.flatnonzero(had_direct)
allowed=np.zeros(N,bool);allowed[rows]=line_eligible[rows,outside_owner[rows]]
# Removing forced evidence returns the frame to ordinary confident/contested processing.
# Убираем принудительное свидетельство: кадр возвращается к обычным уверенным/спорным.
outside_owner[had_direct&~allowed]=3
assert np.all(outside_owner[allowed]==original_outside_owner[allowed])
assert np.all(outside_owner[had_direct&~allowed]==3)
np.save(OUT/'corridor_eligible.npy',allowed)
np.save(OUT/'corridor_suppressed.npy',had_direct&~allowed)

# Replace legacy D completely before running the H/A solver.
# Полностью заменяем старый D перед запуском алгоритма H/A.
dead=filtered.copy()
selected=json.loads((ROOT/'sol/007_clean_anchors/selected_training_config.json').read_text())
progress('021 full inference')
labels,stats,gaps=predict_interval(selected,0,N,collect=True)
original_x=labels=='X';fallback_counts={};graded=np.zeros(N,bool)
def fill_nearest_time(values):
 valid=np.flatnonzero(np.isin(values,['H','A']))
 assert len(valid)>0
 missing=np.flatnonzero(~np.isin(values,['H','A']))
 if len(missing):
  right=np.searchsorted(valid,missing);lo=valid[np.maximum(right-1,0)];hi=valid[np.minimum(right,len(valid)-1)]
  chosen=np.where(missing-lo<=hi-missing,lo,hi)
  values[missing]=values[chosen]
 return values
# Fill only missing labels. Never change a confident anchor or a D decision.
# Заполняем только отсутствующие метки: уверенные якоря и решения D неизменны.
for half,first,last in BLOCKS:
 graded[first:last+1]=True
 nearest_evidence=np.full(last-first+1,'X',dtype='<U1')
 d=nearest[first:last+1];available=np.isfinite(d).any(axis=1)
 winner=np.argmin(d,axis=1)
 nearest_evidence[available]=np.array(['H','A'])[winner[available]]
 forced=outside_owner[first:last+1]!=3
 nearest_evidence[forced]=np.array(['H','A'])[outside_owner[first:last+1][forced]]
 assert np.isin(nearest_evidence,['H','A']).any(), 'No team evidence in entire half'
 half_prior=fill_nearest_time(nearest_evidence.copy())
 counts=dict(anchor_edge_fill=0,nearest_ball_fallback=0,same_half_fallback=0)
 states=dead[first:last+1]
 for a,b in runs_of(states):
  if states[a]:continue
  lo,hi=first+a,first+b;part=labels[lo:hi].copy();missing=part=='X'
  if not missing.any():continue
  if np.isin(part,['H','A']).any():
   counts['anchor_edge_fill']+=int(missing.sum());part=fill_nearest_time(part)
  else:
   evidence=nearest_evidence[a:b].copy()
   if np.isin(evidence,['H','A']).any():
    counts['nearest_ball_fallback']+=len(part);part=fill_nearest_time(evidence)
   else:
    counts['same_half_fallback']+=len(part);part=half_prior[a:b].copy()
  labels[lo:hi]=part
 fallback_counts[str(half)]=counts
labels[~graded]='D'
assert not np.any(labels=='X')
assert np.array_equal(labels[graded]=='D',filtered[graded])
assert np.array_equal(labels[graded&~original_x],predict_interval(selected,0,N)[0][graded&~original_x])
result=pd.DataFrame({'frame':np.arange(N),'label':labels});assert pm.validate_submission(result,N) is None
result.to_csv(OUT/'submission.csv',index=False)
# Exact start-of-half audit, including the second half pre-kickoff coverage.
# Точная проверка начал таймов, включая участок до второго розыгрыша.
audit={str(h):dict(first_frame=a,last_frame=b,first_label=str(labels[a]),X_frames=int((labels[a:b+1]=='X').sum()),initial_10s_X=int((labels[a:a+250]=='X').sum())) for h,a,b in BLOCKS}
(OUT/'coverage.json').write_text(json.dumps(dict(total_X=int((labels=='X').sum()),original_X_in_halves=int((original_x&graded).sum()),fallback_counts=fallback_counts,halves=audit,outside_half_policy='D'),indent=2))
print(json.dumps(audit,indent=2))


In [ ]:
# Evaluate once the main and secondary parameters are frozen.
# Оцениваем после фиксации основного и дополнительного наборов параметров.
public=gt.loc[gt.label.isin(['H','A','D']),'frame'].to_numpy()
def multiclass_scores(prediction,frames):
 y=pm.label_codes(gt.set_index('frame').loc[frames,'label'].to_numpy());p=pm.label_codes(prediction[frames])
 runs=pm.valid_runs(y);cm=pm.scored_confusion(pm.confusion(y,p))
 result=dict(spellF1=pm.spell_scores(y,pm.fill_abstentions(p,runs),runs)['mean_f1'],macroF1=pm.macro_f1(cm),classes={})
 for i,label in enumerate(['H','A','D']):
  precision,recall,f1=pm.per_class_prf(cm,i)
  result['classes'][label]=dict(precision=precision,recall=recall,F1=f1,support=int(cm[i].sum()),predicted=int(cm[:,i].sum()))
 assert np.isclose(result['macroF1'],np.mean([row['F1'] for row in result['classes'].values()]))
 return result
main_metrics=dict(public_5min=multiclass_scores(labels,public),last_minute=multiclass_scores(labels,public[6000:]),
 context='016 fitted on public prefix, D-expansion selected on isolated first 4 minutes; full offline inference; repeatedly viewed last minute')
save_json(OUT/'metrics.json',main_metrics)
# Reuse prepared H/A functions to construct the p95 diagnostic variant without refitting.
# Для диагностического варианта p95 используем готовые H/A-функции без обучения.
parent020=json.loads((ROOT/'sol/020_penalty_gated_corridor/experiment.ipynb').read_text())
tail=''.join(parent020['cells'][2]['source']).split('dead=filtered.copy()')[1]
variant_env=dict(globals());variant_folder=OUT/'variant_p95';variant_folder.mkdir(exist_ok=True)
variant_env.update(OUT=variant_folder,filtered=full_masks['0.95'])
# Copied functions resolve globals in the original namespace; set the shared D barrier too.
# Определённые функции используют исходное пространство имён; обновляем общий барьер D.
main_dead=dead.copy();dead=full_masks['0.95'].copy()
exec('dead=filtered.copy()'+tail,variant_env)
variant_labels=variant_env['labels'];dead=main_dead
variant_metrics=dict(public_5min=multiclass_scores(variant_labels,public),last_minute=multiclass_scores(variant_labels,public[6000:]))
save_json(variant_folder/'metrics.json',variant_metrics)
baseline_labels=pd.read_csv(ROOT/'local_review/experiments/020_penalty_gated_corridor/submission.csv').label.to_numpy()
y_all=gt.label.eq('D').to_numpy();binary_report={}
for name,mask in [('cores017',broadcast_cores),('selected_cores',full_cores),('020',baseline_labels=='D'),('021_p99',full_masks['0.99']),('021_p95',full_masks['0.95'])]:
 binary_report[name]={}
 for window,frames in [('public_5min',public),('last_minute',public[6000:])]:
  entry=binary_scores(y_all[frames],mask[frames]);entry['D_runs']=sum(bool(mask[frames][a]) for a,b in spans(mask[frames]))
  entry['true_D_runs_hit_1s']=int(sum(bool(y_all[frames][a]) and mask[frames][a:b].sum()>=FPS for a,b in spans(y_all[frames])))
  binary_report[name][window]=entry
save_json(OUT/'binary_comparison.json',binary_report)
sequence_rows=[]
for name,prediction in [('020',baseline_labels),('021_p99',labels),('021_p95',variant_labels),('reference',gt.label.to_numpy())]:
 for label in ['H','A','D','X']:
  segment=prediction[public]
  sequence_rows.append(dict(model=name,label=label,sequences=int(sum(segment[a]==label for a,b in spans(segment))),frames=int((segment==label).sum())))
pd.DataFrame(sequence_rows).to_csv(OUT/'sequence_counts_5min.csv',index=False)
assert np.all(~full_cores|filtered)
assert np.array_equal(labels[graded]=='D',filtered[graded])
assert np.array_equal(variant_labels[graded]=='D',full_masks['0.95'][graded])
assert set(labels)<= {'H','A','D'} and set(variant_labels)<= {'H','A','D'}
save_json(OUT/'model_config.json',dict(D_source='broadcast and optional high-confidence 016 cores with constrained expansion',primary_precision_target=.99,
 expansion=selected_expansion['0.99'],secondary_expansion=selected_expansion['0.95'],
 smoothing_frames=25,live_barrier=dict(probability_max=.15,ball_speed_min=3.,player_speed_min=1.,duration_frames=15),
 seed_min_frames=100,seed_gap_frames=10,secondary_seed_probability=.9,secondary_seed_min_frames=25,HA_parent='020_penalty_gated_corridor',HA_config=selected,corridor_min_own_goal_depth_m=16.5,
 outside_half_label='D',X_count=0,selection_range=[START,SPLIT],
 probability_sha256=hashlib.sha256((ROOT/'local_review/experiments/020_penalty_gated_corridor/p_D.npy').read_bytes()).hexdigest()))
progress('completed',candidates=len(configs),X_frames=0)
print(json.dumps(dict(selected_expansion=selected_expansion,metrics=main_metrics,binary=binary_report),indent=2))


In [ ]:
# Aggregate episode diagnostics; never emit restricted tracking rows.
# Агрегированная диагностика эпизодов, без вывода строк ограниченного трекинга.
segments=[]
true_public=gt.set_index('frame').loc[public,'label'].eq('D').to_numpy()
for number,(a,b) in enumerate([(a,b) for a,b in spans(true_public) if true_public[a]],start=1):
 frames=public[a:b]
 segments.append(dict(episode=number,duration_seconds=len(frames)/FPS,
  broadcast_core_frames=int(broadcast_cores[frames].sum()),selected_core_frames=int(full_cores[frames].sum()),
  old_D_frames=int((baseline_labels[frames]=='D').sum()),new_D_frames=int(filtered[frames].sum())))
pd.DataFrame(segments).to_csv(OUT/'public_stop_coverage.csv',index=False)
lines=['# 021 — ядра D и восстановление границ','',
 'Новый алгоритм создаёт D только вокруг автоматических ядер. Сначала проверено расширение broadcast-ядер, затем после обнаружения недостатка покрытия добавлены устойчивые P(D)>=0.9 из 016. Это адаптивная диагностика на ранее исследованном публичном отрезке, не независимая оценка.',
 'Перебрано 122 настройки. Выбор максимального recall при precision>=0.99 (основной) или >=0.95 (вариант) на изолированных первых четырёх минутах. Будущие ядра пятой минуты не участвуют в этом подборе. Однако 016 обучена на публичном префиксе, поэтому подбор оптимистичен.',
 'При полном инференсе используются будущие входные данные внутри тайма, но не эталонные метки. Ядра сохраняются целиком, поэтому ошибочный якорь невозможно исправить расширением.',
 '', '## Выбранные настройки','',json.dumps(selected_expansion,ensure_ascii=False,indent=2),
 '', 'Вес дополнительного движения выбран равным 0: отдельный член для скорости не улучшил выбранную точку на префиксе. Устойчивый LIVE-барьер с движением остаётся частью алгоритма.',
 '', '## Общие метрики (с H/A из 020)','',
 '| Окно | 020 spellF1 | 021 spellF1 | 020 macroF1 | 021 macroF1 |','|---|---:|---:|---:|---:|']
old_metrics=json.loads((ROOT/'local_review/experiments/020_penalty_gated_corridor/metrics.json').read_text())
for window in ['public_5min','last_minute']:
 a=old_metrics[window];b=main_metrics[window]
 lines.append(f"| {window} | {a['spellF1']:.4f} | {b['spellF1']:.4f} | {a['macroF1']:.4f} | {b['macroF1']:.4f} |")
lines+=['','## D отдельно','', '| Вариант | Окно | Precision | Recall | F1 | Ложных D, с | Найдено остановок >=1с |','|---|---|---:|---:|---:|---:|---:|']
for method,windows in binary_report.items():
 for window,m in windows.items():lines.append(f"| {method} | {window} | {m['precision']:.4f} | {m['recall']:.4f} | {m['F1']:.4f} | {m['false_D_seconds']:.2f} | {m['true_D_runs_hit_1s']} |")
lines+=['','## Покрытие четырёх публичных остановок','', '| Эпизод | Длительность, с | Кадров broadcast-ядра | Кадров объединённого ядра | Найдено 020 | Найдено 021 |','|---|---:|---:|---:|---:|---:|']
for row in segments:lines.append(f"| {row['episode']} | {row['duration_seconds']:.2f} | {row['broadcast_core_frames']} | {row['selected_core_frames']} | {row['old_D_frames']} | {row['new_D_frames']} |")
lines+=['','020 сохраняет лучшие общие метрики. 021 — отдельный режим с приоритетом точности D и меньшим recall, а не подтверждённое улучшение полного решения. Одна остановка остаётся без ядра; расширение её не восстановит. X отсутствуют в обоих вариантах.']
(OUT/'REVIEW_RU.md').write_text('\n'.join(lines)+'\n')
# End-to-end invariants: both variants preserve their own seeds and exact D masks.
# Сквозные проверки: каждый вариант сохраняет свои ядра и точное соответствие D-маске.
for target,prediction in [('0.99',labels),('0.95',variant_labels)]:
 mask=full_masks[target];core=all_cores[target]
 assert not np.any(prediction=='X')
 assert np.all(~core|mask)
 assert np.array_equal(prediction[graded]=='D',mask[graded])
 for _,a,b in BLOCKS:
  segment=mask[a:b+1]
  assert all(core[a+x:a+y].any() for x,y in spans(segment) if segment[x])
save_json(OUT/'validation.json',dict(full_prediction_contract=True,no_X=True,seeds_preserved=True,
 each_D_component_has_seed=True,synthetic_boundary_cases_passed=True,prefix_selection_isolated=True))
print('Boundary, coverage and prediction checks passed')
